In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from src.config import *

In [0]:
%sql
create table if not exists projects.ecommerce.dim_products (
    product_sk bigint generated always as identity,
    product_id int,
    title string,
    description string,
    category string,
    price float,
    discount_percentage float,
    rating float,
    stock int,
    brand string,
    sku string,
    weight float,
    width float,
    height float,
    depth float,
    warranty_information string,
    shipping_information string,
    availability_status string,
    return_policy string,
    min_order_quantity int,
    bar_code string,
    qr_code string,
    thumbnail string,
    effective_date date default current_date(),
    expiration_date date default date '9999-12-31',
    is_current boolean default true
)
tblproperties('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
merge into projects.ecommerce.dim_products t
using projects.ecommerce.silver_products s
on t.product_id = s.product_id

-- expires existing products enteries - SCD2
when matched and (
    t.price <> s.price or
    t.discount_percentage <> s.discount_percentage
)
then
update set
    t.expiration_date = date_sub(current_date(), 1),
    t.is_current = false

-- updates existing products enteries - SCD1
when matched and (
    t.title <> s.title or 
    t.description <> s.description or
    t.category <> s.category or
    t.rating <> s.rating or
    t.stock <> s.stock or
    t.brand <> s.brand or
    t.sku <> s.sku or
    t.weight <> s.weight or
    t.width <> s.width or
    t.height <> s.height or
    t.depth <> s.depth or
    t.warranty_information <> s.warranty_information or
    t.shipping_information <> s.shipping_information or
    t.availability_status <> s.availability_status or
    t.return_policy <> s.return_policy or
    t.min_order_quantity <> s.min_order_quantity or
    t.bar_code <> s.bar_code or
    t.qr_code <> s.qr_code or
    t.thumbnail <> s.thumbnail
)
then
update set
    t.title = s.title,
    t.description = s.description,
    t.category = s.category,
    t.rating = s.rating,
    t.stock = s.stock,
    t.brand = s.brand,
    t.sku = s.sku,
    t.weight = s.weight,
    t.width = s.width,
    t.height = s.height,
    t.depth = s.depth,
    t.warranty_information = s.warranty_information,
    t.shipping_information = s.shipping_information,
    t.availability_status = s.availability_status,
    t.return_policy = s.return_policy,
    t.min_order_quantity = s.min_order_quantity,
    t.bar_code = s.bar_code,
    t.qr_code = s.qr_code,
    t.thumbnail = s.thumbnail

-- creates new products enteries
when not matched
then insert (
    t.product_id,
    t.title,
    t.description,
    t.category,
    t.price,
    t.discount_percentage,
    t.rating,
    t.stock,
    t.brand,
    t.sku,
    t.weight,
    t.width,
    t.height,
    t.depth,
    t.warranty_information,
    t.shipping_information,
    t.availability_status,
    t.return_policy,
    t.min_order_quantity,
    t.bar_code,
    t.qr_code,
    t.thumbnail
)
values (
    s.product_id,
    s.title,
    s.description,
    s.category,
    s.price,
    s.discount_percentage,
    s.rating,
    s.stock,
    s.brand,
    s.sku,
    s.weight,
    s.width,
    s.height,
    s.depth,
    s.warranty_information,
    s.shipping_information,
    s.availability_status,
    s.return_policy,
    s.min_order_quantity,
    s.bar_code,
    s.qr_code,
    s.thumbnail
);

In [0]:
%sql
-- creates new enteries for SCD2 expired products
insert into projects.ecommerce.dim_products (
    product_id,
    title,
    description,
    category,
    price,
    discount_percentage,
    rating,
    stock,
    brand,
    sku,
    weight,
    width,
    height,
    depth,
    warranty_information,
    shipping_information,
    availability_status,
    return_policy,
    min_order_quantity,
    bar_code,
    qr_code,
    thumbnail
)
select
    s.product_id,
    s.title,
    s.description,
    s.category,
    s.price,
    s.discount_percentage,
    s.rating,
    s.stock,
    s.brand,
    s.sku,
    s.weight,
    s.width,
    s.height,
    s.depth,
    s.warranty_information,
    s.shipping_information,
    s.availability_status,
    s.return_policy,
    s.min_order_quantity,
    s.bar_code,
    s.qr_code,
    s.thumbnail
from projects.ecommerce.silver_products s
join projects.ecommerce.dim_products t on t.product_id = s.product_id
where t.expiration_date = current_date() and t.is_current = false;